# Task 4 — forward model of surface displacement

For each nodal plane, a rectangular uniform-slip Okada (1992) dislocation
via okada4py, with fault dimensions from Wells & Coppersmith (1994).
This is the quantity the pipeline exists to produce: is this event
InSAR-relevant? Parameters in `auto_tdmt.cfg` §4.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
# work in a scratch archive so the real events/ are untouched
os.environ.setdefault("AUTO_TDMT_EVENTS", str(Path.home() / "work" / "proj_tdmt_NZ" / "notebook_runs"))
import config
from config import P            # every tunable, from auto_tdmt.cfg
EVENT = "2026p669681"
print("parameters from", P.source)

In [ ]:
print(json.dumps(P.as_dict()["forward"], indent=2))

## 4.1 From an archived solution

In [ ]:
import okada_forward
cands = sorted(Path(config.REPO_DIR, "events").glob(f"{EVENT}*/solution.json")) or \
        sorted(config.EVENTS_DIR.glob(f"{EVENT}*/solution.json"))
sol = json.loads(cands[0].read_text())
fwd = okada_forward.forward_both_planes(sol)
print(f"peak |u| {fwd['peak_abs_m']*100:.3f} cm  detectable: {fwd['detectable']}")
for pl in ("plane1", "plane2"):
    print(pl, fwd[pl]["fault"])

## 4.2 The displacement field

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, pl in zip(axes, ("plane1", "plane2")):
    g = fwd[pl]
    im = ax.pcolormesh(g["x_km"], g["y_km"], g["uz_m"] * 100, cmap="RdBu_r",
                       vmin=-abs(g["uz_m"]).max()*100, vmax=abs(g["uz_m"]).max()*100, shading="auto")
    ax.set_title(f"{pl}: strike {g['fault']['strike']:.0f} dip {g['fault']['dip']:.0f} rake {g['fault']['rake']:.0f}")
    ax.set_aspect("equal"); ax.set_xlabel("east (km)"); ax.set_ylabel("north (km)")
    plt.colorbar(im, ax=ax, label="vertical (cm)")
plt.tight_layout(); plt.show()

## What to check
- Rigidity, the Wells & Coppersmith coefficients and the grid are all in the cfg.
- The publish gate uses `peak_abs_m` against `publish.minDisplacementM`.